In [ ]:
# Prepare paired dataset in Google Drive for this repo
# - Expects source structure like MyDrive/LUTor/Adobe5kJPG/{raw,a,b,c,d,e}
# - Produces MyDrive/LUTor/datasets/Adobe5kJPG_<style>/{train,val}/{input,gt}
# - Uses symlinks when possible (fast, no extra storage); falls back to copy on error

import os
import shutil
from pathlib import Path
from typing import Tuple

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)

# User-configurable
STYLE = 'c'  # choose from 'a','b','c','d','e'
VAL_RATIO = 0.05  # 5% for validation, change as needed
RANDOM_SEED = 42

# Paths
DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.home() / 'MyDrive'
PROJECT_ROOT = DRIVE_ROOT / 'LUTor'
SRC_ROOT = PROJECT_ROOT / 'Adobe5kJPG'
DATASETS_ROOT = PROJECT_ROOT / 'datasets'
DST_ROOT = DATASETS_ROOT / f'Adobe5kJPG_{STYLE}'

SRC_RAW = SRC_ROOT / 'raw'
SRC_STYLE = SRC_ROOT / STYLE

for p in [PROJECT_ROOT, SRC_ROOT, SRC_RAW, SRC_STYLE]:
    if not p.exists():
        raise FileNotFoundError(f'Missing path: {p}. Please ensure your Drive has the expected structure.')

# Collect image files (common extensions)
EXTS = {'.jpg','.jpeg','.png','.tif','.tiff'}
raw_files = sorted([p for p in SRC_RAW.iterdir() if p.suffix.lower() in EXTS])
style_files = sorted([p for p in SRC_STYLE.iterdir() if p.suffix.lower() in EXTS])

# Build name -> path map and intersect
raw_map = {f.name: f for f in raw_files}
style_map = {f.name: f for f in style_files}
common_names = sorted(set(raw_map.keys()) & set(style_map.keys()))
if not common_names:
    raise RuntimeError('No paired files found between raw/ and style folder. Filenames must match.')

# Split into train/val deterministically
import random
random.seed(RANDOM_SEED)
names = common_names.copy()
random.shuffle(names)
val_count = max(1, int(len(names) * VAL_RATIO))
val_names = set(names[:val_count])
train_names = set(names[val_count:])

# Create destination structure
for sub in ['train/input','train/gt','val/input','val/gt']:
    (DST_ROOT / sub).mkdir(parents=True, exist_ok=True)


def _link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        # Prefer relative symlink for portability
        rel_src = os.path.relpath(src, start=dst.parent)
        dst.symlink_to(rel_src)
    except Exception:
        shutil.copy2(src, dst)

# Populate
for split, name_set in [('train', train_names), ('val', val_names)]:
    for name in sorted(name_set):
        src_in = raw_map[name]
        src_gt = style_map[name]
        dst_in = DST_ROOT / split / 'input' / name
        dst_gt = DST_ROOT / split / 'gt' / name
        _link_or_copy(src_in, dst_in)
        _link_or_copy(src_gt, dst_gt)

print(f'Dataset prepared at: {DST_ROOT}')
print('Structure: train/val with input/gt. You can now point training --data_root to this path.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

# LoR-IA-3DLUT • Colab Quickstart

按顺序执行每个单元即可开始训练。

In [1]:
!nvidia-smi
import os

Sun Oct 26 07:24:04 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             33W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/LUTor/lor_ia3dlut_starter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/LUTor/lor_ia3dlut_starter


In [3]:
!pip install -r requirements.txt

## 准备数据
将你的数据整理为 `/dataset_root/{train,val}/{input,gt}` 结构，然后设置到下面变量。

In [4]:
DATA_ROOT = '/content/drive/MyDrive/LUTor/datasets/Adobe5kJPG_c'  # 修改成你的路径
WORK_DIR = '/content/drive/MyDrive/LUTor/lor_ia3dlut_starter/runs/fivek_lor'

In [5]:
import yaml
with open("config/default.yaml") as f:
    cfg = yaml.safe_load(f)
print(cfg["loss"])
print({k:type(v).__name__ for k,v in cfg["loss"].items()})

{'l1': 1.0, 'de2000': 0.0, 'tv': 0.0001, 'dl2': 0.0001, 'lpips': 0.0, 'alpha_l2': 0.0001}
{'l1': 'float', 'de2000': 'float', 'tv': 'float', 'dl2': 'float', 'lpips': 'float', 'alpha_l2': 'float'}


In [ ]:
#check dataset preparation
from data.paired_folder import PairedFolderDataset
ds = PairedFolderDataset(root=DATA_ROOT, split="train", in_dir="input", gt_dir="gt", patch=0, augment=False)
for i in range(5):
    s = ds[i]
    print(s["name"], (s["img_in"]-s["img_gt"]).abs().mean().item())

a0001-jmac_DSC1459.jpg 0.23830972611904144
a0002-dgw_005.jpg 0.08535352349281311
a0003-NKIM_MG_8178.jpg 0.02693273313343525
a0004-jmac_MG_1384.jpg 0.030018212273716927
a0005-jn_2007_05_10__564.jpg 0.07374764233827591


In [ ]:
import torch
import yaml
from core.core_lut import LoRIA3DLUT
from data.paired_folder import PairedFolderDataset
from losses.delta_e import delta_e_2000_srgb

# 1. 加载配置和模型
ckpt_path = "/content/drive/MyDrive/LUTor/lor_ia3dlut_starter/runs/fivek_lor/best.ckpt"  # 或者 last.ckpt，或你训练到16000步左右的ckpt
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device)
cfg = ckpt["cfg"]

G = cfg["model"]["G"]
K = cfg["model"]["K"]
R = cfg["model"]["R"]

model = LoRIA3DLUT(G=G, K=K, R=R).to(device)
model.load_state_dict(ckpt["state_dict"], strict=True)
model.eval()

print(f"Model loaded: G={G}, K={K}, R={R}")

# 2. 加载数据集
root = "/content/drive/MyDrive/LUTor/datasets/Adobe5kJPG_c"  # 改成你的数据集路径
ds = PairedFolderDataset(
    root=root,
    split="train",
    in_dir="input",
    gt_dir="gt",
    exts=tuple(cfg["data"]["ext"]),
    patch=512,
    augment=False
)

print(f"Dataset loaded: {len(ds)} samples")

# 3. 测试几个样本
num_test = 5
for i in range(min(num_test, len(ds))):
    sample = ds[i]
    img_lr = sample["img_lr"].unsqueeze(0).to(device)  # [1,3,256,256]
    img_in = sample["img_in"].unsqueeze(0).to(device)  # [1,3,512,512]
    img_gt = sample["img_gt"].unsqueeze(0).to(device)

    with torch.no_grad():
        pred, _ = model(img_lr, img_in)

        pred_min = pred.min().item()
        pred_max = pred.max().item()
        pred_mean = pred.mean().item()

        print(f"\n[Sample {i}] {sample['name']}")
        print(f"  pred range: [{pred_min:.4f}, {pred_max:.4f}], mean={pred_mean:.4f}")

        # 检查是否超出 [0,1]
        out_of_range = (pred < 0).any() or (pred > 1).any()
        if out_of_range:
            print(f"  ⚠️  pred 超出 [0,1] 范围!")

        # 测试 ΔE2000（不clamp）
        try:
            de_no_clamp = delta_e_2000_srgb(pred, img_gt)
            de_mean = de_no_clamp.mean().item()
            has_nan = torch.isnan(de_no_clamp).any().item()
            has_inf = torch.isinf(de_no_clamp).any().item()
            print(f"  ΔE2000 (no clamp): mean={de_mean:.4f}, has_nan={has_nan}, has_inf={has_inf}")
        except Exception as e:
            print(f"  ❌ ΔE2000 (no clamp) 崩溃: {e}")

        # 测试 ΔE2000（clamp到[0,1]）
        try:
            de_clamped = delta_e_2000_srgb(pred.clamp(0,1), img_gt.clamp(0,1))
            de_mean_c = de_clamped.mean().item()
            has_nan_c = torch.isnan(de_clamped).any().item()
            print(f"  ΔE2000 (clamped):  mean={de_mean_c:.4f}, has_nan={has_nan_c}")
        except Exception as e:
            print(f"  ❌ ΔE2000 (clamped) 崩溃: {e}")

print("\n✅ 验证完成")

Loading checkpoint: /content/drive/MyDrive/LUTor/lor_ia3dlut_starter/runs/fivek_lor/best.ckpt
Model loaded: G=33, K=3, R=8
Dataset loaded: 4750 samples

[Sample 0] a0002-dgw_005.jpg
  pred range: [0.0353, 0.8353], mean=0.3825
  ΔE2000 (no clamp): mean=7.8615, has_nan=False, has_inf=False
  ΔE2000 (clamped):  mean=7.8615, has_nan=False

[Sample 1] a0003-NKIM_MG_8178.jpg
  pred range: [-0.0000, 0.4902], mean=0.1411
  ⚠️  pred 超出 [0,1] 范围!
  ΔE2000 (no clamp): mean=4.0914, has_nan=False, has_inf=False
  ΔE2000 (clamped):  mean=4.0914, has_nan=False

[Sample 2] a0004-jmac_MG_1384.jpg
  pred range: [-0.0000, 0.9451], mean=0.3760
  ⚠️  pred 超出 [0,1] 范围!
  ΔE2000 (no clamp): mean=7.1754, has_nan=False, has_inf=False
  ΔE2000 (clamped):  mean=7.1754, has_nan=False

[Sample 3] a0005-jn_2007_05_10__564.jpg
  pred range: [0.0078, 1.0000], mean=0.3251
  ΔE2000 (no clamp): mean=8.2806, has_nan=False, has_inf=False
  ΔE2000 (clamped):  mean=8.2806, has_nan=False

[Sample 4] a0006-IMG_2787.jpg
  pred

In [ ]:
# 在 Colab 里运行
import torch
from data.paired_folder import PairedFolderDataset
from core.core_lut import LoRIA3DLUT
from utils.metrics import psnr

root = "/content/drive/MyDrive/LUTor/datasets/Adobe5kJPG_c"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 检查验证集大小
ds_val = PairedFolderDataset(root, "val", "input", "gt",
                              exts=(".jpg",".jpeg",".png",".tif",".tiff"),
                              patch=0, augment=False)
print(f"✅ 验证集图片数: {len(ds_val)}")

# 2. 检查前几张图的 input vs gt 差异
print("\n验证集前5张的 input-gt 差异:")
for i in range(min(5, len(ds_val))):
    s = ds_val[i]
    mae = (s["img_in"] - s["img_gt"]).abs().mean().item()
    print(f"  {s['name']}: MAE={mae:.4f}")

# 3. 加载模型，看"恒等输入"的 PSNR 是多少
ckpt = torch.load("runs/fivek_lor/best.ckpt", map_location=device)
cfg = ckpt["cfg"]
model = LoRIA3DLUT(G=cfg["model"]["G"], K=cfg["model"]["K"], R=cfg["model"]["R"]).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()

with torch.no_grad():
    s = ds_val[0]
    img_lr = s["img_lr"].unsqueeze(0).to(device)
    img_in = s["img_in"].unsqueeze(0).to(device)
    img_gt = s["img_gt"].unsqueeze(0).to(device)

    # 模型预测
    pred, _ = model(img_lr, img_in)
    psnr_pred = psnr(pred.clamp(0,1), img_gt.clamp(0,1)).item()

    # 恒等输出（直接用 input）
    psnr_identity = psnr(img_in.clamp(0,1), img_gt.clamp(0,1)).item()

    print(f"\n第一张图的 PSNR:")
    print(f"  模型输出: {psnr_pred:.2f}")
    print(f"  恒等输出: {psnr_identity:.2f}")
    print(f"  差异: {psnr_pred - psnr_identity:.2f}")

✅ 验证集图片数: 250

验证集前5张的 input-gt 差异:
  a0001-jmac_DSC1459.jpg: MAE=0.2383
  a0016-jmac_MG_0795.jpg: MAE=0.0504
  a0019-jmac_MG_0653.jpg: MAE=0.0984
  a0023-07-06-02-at-15h06m48-s_MG_1489.jpg: MAE=0.0590
  a0043-07-11-27-at-12h09m46s-_MG_7307.jpg: MAE=0.0586

第一张图的 PSNR:
  模型输出: 11.92
  恒等输出: 11.92
  差异: 0.00


In [6]:
!python train.py \
  --data.root "$DATA_ROOT" \
  --work_dir "$WORK_DIR" \
  --cfg config/default.yaml

流式输出内容被截断，只能显示最后 5000 行内容。
[ 50260/100000] loss=0.0261 psnr=26.26 de2000=3.247 amax=0.645 dnorm=0.0221 lr=5.46e-05 time=280.5m
[ 50270/100000] loss=0.0521 psnr=24.30 de2000=5.457 amax=0.650 dnorm=0.0206 lr=5.46e-05 time=280.6m
[ 50280/100000] loss=0.0900 psnr=20.67 de2000=9.098 amax=0.703 dnorm=0.0732 lr=5.46e-05 time=280.6m
[ 50290/100000] loss=0.2416 psnr=10.60 de2000=30.283 amax=0.650 dnorm=0.0181 lr=5.46e-05 time=280.7m
[ 50300/100000] loss=0.1932 psnr=14.20 de2000=18.555 amax=0.683 dnorm=0.0340 lr=5.46e-05 time=280.7m
[ 50310/100000] loss=0.0556 psnr=22.38 de2000=10.508 amax=0.661 dnorm=0.0280 lr=5.46e-05 time=280.7m
[ 50320/100000] loss=0.0920 psnr=20.32 de2000=8.866 amax=0.666 dnorm=0.0176 lr=5.45e-05 time=280.8m
[ 50330/100000] loss=0.0208 psnr=31.64 de2000=4.044 amax=0.694 dnorm=0.0654 lr=5.45e-05 time=280.8m
[ 50340/100000] loss=0.0884 psnr=19.86 de2000=8.625 amax=0.679 dnorm=0.0458 lr=5.45e-05 time=280.9m
[ 50350/100000] loss=0.0535 psnr=22.34 de2000=6.814 amax=0.651 dnorm=0

## 验证与可视化

In [7]:

!python evaluate.py \
  --data.root "$DATA_ROOT/val" \
  --ckpt "$WORK_DIR/best.ckpt" \
  --out_dir "$WORK_DIR/val_vis"

{'N': 250, 'PSNR_mean': 23.02079951095581, 'PSNR_std': 4.551711800769015, 'DeltaE2000_mean': 7.767336657047272, 'DeltaE2000_std': 4.152748228366088}


In [8]:
import torch
ckpt = torch.load("/content/drive/MyDrive/LUTor/lor_ia3dlut_starter/runs/fivek_lor/best.ckpt", map_location="cpu")
print(ckpt.keys())  # dict_keys(['state_dict','optimizer','cfg','iter','best'])
print("iter:", ckpt["iter"], "best:", ckpt["best"])
# 看权重名和形状（前10个）
for i,(k,v) in enumerate(ckpt["state_dict"].items()):
    print(k, tuple(v.shape))
    # if i>=9: break

dict_keys(['state_dict', 'optimizer', 'cfg', 'iter', 'best'])
iter: 76000 best: 23.02079963684082
bases (8, 33, 33, 33, 3)
weight_pred.backbone.0.weight (16, 3, 3, 3)
weight_pred.backbone.0.bias (16,)
weight_pred.backbone.2.weight (32, 16, 3, 3)
weight_pred.backbone.2.bias (32,)
weight_pred.fc.weight (8, 32)
weight_pred.fc.bias (8,)
resid_pred.enc.0.weight (16, 3, 3, 3)
resid_pred.enc.0.bias (16,)
resid_pred.enc.2.weight (32, 16, 3, 3)
resid_pred.enc.2.bias (32,)
resid_pred.fc_u.weight (264, 32)
resid_pred.fc_u.bias (264,)
resid_pred.fc_v.weight (264, 32)
resid_pred.fc_v.bias (264,)
resid_pred.fc_w.weight (264, 32)
resid_pred.fc_w.bias (264,)
resid_pred.fc_c.weight (24, 32)
resid_pred.fc_c.bias (24,)
